# Frozen shape probe: localizing the blue-square failure

**Question:** Does shape information survive from the CNN feature map into the projected TinyCLIP image embedding for held-out `blue square`, relative to the successful held-out control `red triangle`?

The linear probe is trained only on green circles, triangles, and squares. Color is therefore constant and cannot serve as a shortcut. TinyCLIP remains frozen throughout probing.

**Committed prediction under the projection-loss hypothesis**

| Representation | Red triangle | Blue square |
|---|---:|---:|
| CNN pre-projection | high | high |
| CLIP projected | high | low |

In [1]:
import pandas as pd
from torch.utils.data import DataLoader

from clip_repro.data import (
    ShapeDataset,
    all_combinations,
    build_splits,
    build_tokenizer,
    make_collate_fn,
)
from clip_repro.evaluation import (
    evaluate_retrieval,
    prediction_counts,
    probe_result_rows,
    run_shape_probe_experiment,
    summarize_pair,
)
from clip_repro.train import TrainConfig, train_model
from clip_repro.utils import get_device, set_seed

## Configuration

Seed 2 was selected because the previous five-seed replication identified it as a clear blue-square retrieval failure. It is a diagnostic case, not a population estimate.

In [2]:
HELD_OUT_PAIRS = {
    ("red", "triangle"),
    ("blue", "square"),
}
MODEL_SEED = 2
RETRIEVAL_EVAL_SEED = 12345
RETRIEVAL_SAMPLES_PER_COMBO = 20
PROBE_TRAIN_SAMPLES_PER_COMBO = 100
PROBE_TEST_SAMPLES_PER_COMBO = 100
TRAIN_CONFIG = TrainConfig(epochs=500, learning_rate=3e-4)
device = get_device()
device

device(type='mps')

## Construct and verify the held-out split

In [3]:
train_combinations, test_combinations = build_splits(HELD_OUT_PAIRS)
candidate_combinations = all_combinations()
tokenizer = build_tokenizer(candidate_combinations)

train_pairs = {combo[:2] for combo in train_combinations}
test_pairs = {combo[:2] for combo in test_combinations}
green_probe_combinations = [
    combo for combo in train_combinations if combo[0] == "green"
]

assert len(train_combinations) == 42
assert len(test_combinations) == 12
assert HELD_OUT_PAIRS.isdisjoint(train_pairs)
assert test_pairs == HELD_OUT_PAIRS
assert len(green_probe_combinations) == 18
assert {combo[1] for combo in green_probe_combinations} == {
    "circle", "triangle", "square"
}

pd.Series({
    "TinyCLIP train combinations": len(train_combinations),
    "held-out combinations": len(test_combinations),
    "green probe combinations": len(green_probe_combinations),
})

TinyCLIP train combinations    42
held-out combinations          12
green probe combinations       18
dtype: int64

## Train the diagnostic TinyCLIP model

This recreates the selected seed on the exact split above. The stochastic renderer produces new pixels each epoch.

In [4]:
train_loader = DataLoader(
    ShapeDataset(train_combinations),
    batch_size=len(train_combinations),
    shuffle=True,
    collate_fn=make_collate_fn(tokenizer),
)

probe_model, training_history = train_model(
    tokenizer,
    train_loader,
    device,
    TRAIN_CONFIG,
    seed=MODEL_SEED,
    print_every=50,
)

pd.DataFrame(training_history).tail(1)

Epoch    1 | Loss 3.9518 | I2T 0.024 | T2I 0.024 | tau 0.0700
Epoch   50 | Loss 0.5861 | I2T 0.881 | T2I 0.929 | tau 0.0694
Epoch  100 | Loss 0.1626 | I2T 1.000 | T2I 1.000 | tau 0.0685
Epoch  150 | Loss 0.0629 | I2T 1.000 | T2I 1.000 | tau 0.0678
Epoch  200 | Loss 0.0374 | I2T 1.000 | T2I 1.000 | tau 0.0673
Epoch  250 | Loss 0.0301 | I2T 1.000 | T2I 1.000 | tau 0.0669
Epoch  300 | Loss 0.0253 | I2T 1.000 | T2I 1.000 | tau 0.0666
Epoch  350 | Loss 0.0208 | I2T 1.000 | T2I 1.000 | tau 0.0662
Epoch  400 | Loss 0.0181 | I2T 1.000 | T2I 1.000 | tau 0.0659
Epoch  450 | Loss 0.0160 | I2T 1.000 | T2I 1.000 | tau 0.0656
Epoch  500 | Loss 0.0148 | I2T 1.000 | T2I 1.000 | tau 0.0653


,epoch,loss,i2t,t2i,temperature
499,500,0.014771,1.0,1.0,0.065266


## Confirm the retrieval phenomenon before localizing it

If this seed does not reproduce strong red-triangle retrieval and weak blue-square retrieval, it is not the intended diagnostic case and the probe should not be overinterpreted.

In [5]:
set_seed(RETRIEVAL_EVAL_SEED)
retrieval = evaluate_retrieval(
    probe_model,
    test_combinations,
    candidate_combinations,
    tokenizer,
    device,
    RETRIEVAL_SAMPLES_PER_COMBO,
)

retrieval_check = {}
for pair in sorted(HELD_OUT_PAIRS):
    retrieval_check[pair] = {
        **summarize_pair(retrieval, pair),
        "top_predictions": prediction_counts(retrieval, pair).most_common(3),
    }
retrieval_check

{('blue', 'square'): {'accuracy': 0,
  'color_margin': 0.44684073760484655,
  'shape_margin': -0.16610577702522278,
  'top_predictions': [('a small blue circle on the left', 20),
   ('a small blue circle in the center', 20),
   ('a small blue circle on the right', 20)]},
 ('red', 'triangle'): {'accuracy': 1,
  'color_margin': 0.2886069412032763,
  'shape_margin': 0.1296360065539678,
  'top_predictions': [('a small red triangle on the left', 20),
   ('a small red triangle in the center', 20),
   ('a small red triangle on the right', 20)]}}

## Run the green-only frozen linear probes

The reusable function validates the split, freezes every TinyCLIP parameter, renders balanced green-only probe-training data, extracts both representations from identical images, and tests each probe separately on fresh red-triangle and blue-square images.

In [6]:
probe_results = run_shape_probe_experiment(
    model=probe_model,
    train_combinations=train_combinations,
    test_combinations=test_combinations,
    device=device,
    held_out_pairs=HELD_OUT_PAIRS,
    train_samples_per_combo=PROBE_TRAIN_SAMPLES_PER_COMBO,
    test_samples_per_combo=PROBE_TEST_SAMPLES_PER_COMBO,
    probe_epochs=300,
)

assert not any(parameter.requires_grad for parameter in probe_model.parameters())

In [7]:
probe_summary = pd.DataFrame(probe_result_rows(probe_results))[
    [
        "stage",
        "train_accuracy",
        "red_triangle_accuracy",
        "blue_square_accuracy",
    ]
]
probe_summary

,stage,train_accuracy,red_triangle_accuracy,blue_square_accuracy
0,pre_projection,1.0,0.518333,0.031667
1,projected,1.0,1.000000,0.000000


## Optional confusion matrices

Rows are true shapes; columns are predicted shapes. Since each held-out pair has one true shape, its nonzero row shows which alternative absorbs the errors.

In [8]:
shape_order = probe_results["shape_order"]

for stage, stage_result in probe_results["stages"].items():
    for pair, pair_result in stage_result["pairs"].items():
        print(f"{stage}: {pair} | predictions={dict(pair_result['prediction_counts'])}")
        display(
            pd.DataFrame(
                pair_result["confusion_matrix"],
                index=shape_order,
                columns=shape_order,
            ).rename_axis(index="true", columns="predicted")
        )

pre_projection: ('blue', 'square') | predictions={'circle': 243, 'triangle': 338, 'square': 19}


predicted,circle,triangle,square
true,,,
circle,0,0,0
triangle,0,0,0
square,243,338,19


pre_projection: ('red', 'triangle') | predictions={'circle': 157, 'triangle': 311, 'square': 132}


predicted,circle,triangle,square
true,,,
circle,0,0,0
triangle,157,311,132
square,0,0,0


projected: ('blue', 'square') | predictions={'circle': 600}


predicted,circle,triangle,square
true,,,
circle,0,0,0
triangle,0,0,0
square,600,0,0


projected: ('red', 'triangle') | predictions={'triangle': 600}


predicted,circle,triangle,square
true,,,
circle,0,0,0
triangle,0,600,0
square,0,0,0


## Interpretation

- **Pre-projection high; projected blue-square low:** supports composition-dependent information loss or distortion in the image projection.
- **Blue-square low at both stages:** suggests the unusual shape representation already exists in CNN features.
- **Both blue-square probes high while retrieval remains poor:** weakens projection loss and motivates joint image–text geometry analysis.
- **Red-triangle low too:** weakens a blue-square-specific explanation and raises a probe-validity or broader cross-color-transfer concern.

Before changing the model, write down: What happened relative to the prediction? Which hypothesis weakened? What is the cheapest next measurement that could change your belief?